# Labs64 Payment Gateway: Noop Provider Flow

This notebook demonstrates a full local payment flow using the built-in `noop` payment provider.

It covers:

1. Checking the Payment Gateway health endpoint.
2. Listing available payment definitions.
3. Creating or updating the tenant `noop` payment provider.
4. Creating a payment.
5. Executing `/pay` with an `Idempotency-Key`.
6. Reading the payment and payment transactions.
7. Optionally reading RabbitMQ debug queue messages via the Management API.

Before running this notebook, start the local stack and make sure the application is available at `http://localhost:8080`.

## Local Stack

Build the backend jar, then start the dev compose stack from the repository root:

```bash
mvn clean package
docker compose -f docker-compose.dev.yml up -d
```

The notebook uses the temporary development auth headers:

- `X-Tenant-ID`
- `X-Scopes`

These headers are local-development only and should be removed when JWT validation is implemented.

In [10]:
import json
import time
import uuid
from pprint import pprint

import requests

BASE_URL = "http://localhost:8080"
TENANT_ID = "demo-tenant"

SCOPES = " ".join([
    "payment-provider:read",
    "payment-provider:write",
    "payment:read",
    "payment:write",
    "payment:pay",
    "payment-transaction:read",
])

HEADERS = {
    "X-Tenant-ID": TENANT_ID,
    "X-Scopes": SCOPES,
    "Content-Type": "application/json",
    "Accept": "application/json",
}

def pretty(value):
    print(json.dumps(value, indent=2, sort_keys=True))

def request(method, path, **kwargs):
    headers = dict(HEADERS)
    headers.update(kwargs.pop("headers", {}) or {})
    response = requests.request(method, f"{BASE_URL}{path}", headers=headers, timeout=30, **kwargs)
    print(f"{method} {path} -> {response.status_code}")
    if response.content:
        try:
            pretty(response.json())
        except ValueError:
            print(response.text)
    response.raise_for_status()
    return response.json() if response.content else None

## 1. Health Check

In [11]:
health = requests.get(f"{BASE_URL}/actuator/health", timeout=10)
print(health.status_code)
pretty(health.json())
health.raise_for_status()

200
{
  "groups": [
    "liveness",
    "readiness"
  ],
  "status": "UP"
}


## 2. List Available Payment Definitions

`GET /payment-definitions` is public in local development. The `noop` provider should be present.

In [12]:
definitions = request("GET", "/payment-definitions")

providers = {item["provider"] for item in definitions.get("items", [])}
assert "noop" in providers, "The noop payment definition must be configured in application.yml"

GET /payment-definitions -> 200
{
  "items": [
    {
      "description": "Credit/Debit Card via Stripe",
      "icon": null,
      "name": "Stripe",
      "provider": "stripe",
      "recurring": true,
      "supportedCountries": [
        "US",
        "GB",
        "DE"
      ],
      "supportedCurrencies": [
        "USD",
        "EUR",
        "GBP"
      ]
    },
    {
      "description": "PayPal Checkout",
      "icon": null,
      "name": "PayPal",
      "provider": "paypal",
      "recurring": false,
      "supportedCountries": [
        "US",
        "DE"
      ],
      "supportedCurrencies": [
        "USD",
        "EUR"
      ]
    },
    {
      "description": "Test payment provider - always succeeds",
      "icon": null,
      "name": "No-Op (Test)",
      "provider": "noop",
      "recurring": false,
      "supportedCountries": [
        "US",
        "DE"
      ],
      "supportedCurrencies": [
        "USD",
        "EUR"
      ]
    }
  ]
}


## 3. Create Or Update Tenant Payment Provider

`noop` does not require PSP configuration, so the request body omits `config`.

In [13]:
noop_provider_body = {
    "active": True,
    "name": "Noop Demo Provider",
    "description": "Local test provider that succeeds without charging a real PSP."
}

create_response = requests.post(
    f"{BASE_URL}/payment-providers/noop",
    headers=HEADERS,
    json=noop_provider_body,
    timeout=30,
)

print(f"POST /payment-providers/noop -> {create_response.status_code}")
if create_response.status_code == 409:
    print("noop provider already exists; updating it instead")
    provider = request("PATCH", "/payment-providers/noop", json=noop_provider_body)
else:
    if create_response.content:
        pretty(create_response.json())
    create_response.raise_for_status()
    provider = create_response.json()

assert provider["id"] == "noop"
assert provider["active"] is True

POST /payment-providers/noop -> 409
noop provider already exists; updating it instead
PATCH /payment-providers/noop -> 200
{
  "active": true,
  "config": {},
  "createdAt": "2026-06-25T12:39:35.921079Z",
  "description": "Local test provider that succeeds without charging a real PSP.",
  "icon": null,
  "id": "noop",
  "name": "Noop Demo Provider",
  "recurring": null,
  "updatedAt": "2026-06-25T12:39:35.921199Z"
}


## 4. Create Payment

Amounts are integer minor units. `grossAmount=3000` means `30.00 USD`.

In [14]:
payment_request = {
    "provider": "noop",
    "purchaseOrder": {
        "currency": "USD",
        "items": [
            {
                "name": "Widget Pro",
                "description": "Demo product paid through the noop provider",
                "sku": "WIDGET-PRO-DEMO",
                "price": 3000,
                "quantity": 1
            }
        ],
        "netAmount": 3000,
        "grossAmount": 3000,
        "taxAmount": 0
    },
    "billingInfo": {
        "firstName": "Jane",
        "lastName": "Doe",
        "email": "jane.doe@example.com",
        "country": "US",
        "city": "San Francisco",
        "address1": "123 Market St",
        "postalCode": "94105"
    },
    "extra": {
        "source": "jupyter-noop-example",
        "externalOrderId": f"demo-order-{uuid.uuid4()}"
    }
}

payment = request("POST", "/payments", json=payment_request)
payment_id = payment["id"]

assert payment["provider"] == "noop"
assert payment["status"] == "READY"
payment_id

POST /payments -> 201
{
  "billingInfo": {
    "address1": "123 Market St",
    "address2": null,
    "city": "San Francisco",
    "country": "US",
    "email": "jane.doe@example.com",
    "firstName": "Jane",
    "lastName": "Doe",
    "phone": null,
    "postalCode": "94105",
    "state": null,
    "vatId": null
  },
  "createdAt": "2026-06-26T11:07:39.235963Z",
  "extra": {
    "externalOrderId": "demo-order-4a097865-5579-4278-adec-5d5e83ed9553",
    "source": "jupyter-noop-example"
  },
  "id": "84465f35-67b2-4b48-8900-4366fd400854",
  "lastPaymentAt": null,
  "nextPaymentAt": null,
  "provider": "noop",
  "purchaseOrder": {
    "currency": "USD",
    "grossAmount": 3000,
    "items": [
      {
        "description": "Demo product paid through the noop provider",
        "extra": null,
        "image": null,
        "name": "Widget Pro",
        "price": 3000,
        "quantity": 1,
        "sku": "WIDGET-PRO-DEMO",
        "tax": null,
        "uom": null,
        "url": null
    

'84465f35-67b2-4b48-8900-4366fd400854'

## 5. Execute Payment

`/pay` requires an `Idempotency-Key`. The same key can be retried with the same request body and should return the cached response.

In [15]:
idempotency_key = str(uuid.uuid4())

execute_response = request(
    "POST",
    f"/payments/{payment_id}/pay",
    headers={"Idempotency-Key": idempotency_key},
)

assert execute_response["payment"]["status"] == "CLOSED"
assert execute_response["paymentTransaction"]["status"] == "SUCCESS"

payment_transaction_id = execute_response["paymentTransaction"]["id"]
payment_transaction_id

POST /payments/84465f35-67b2-4b48-8900-4366fd400854/pay -> 200
{
  "nextAction": null,
  "payment": {
    "billingInfo": {
      "address1": "123 Market St",
      "address2": null,
      "city": "San Francisco",
      "country": "US",
      "email": "jane.doe@example.com",
      "firstName": "Jane",
      "lastName": "Doe",
      "phone": null,
      "postalCode": "94105",
      "state": null,
      "vatId": null
    },
    "createdAt": "2026-06-26T11:07:39.235963Z",
    "extra": {
      "externalOrderId": "demo-order-4a097865-5579-4278-adec-5d5e83ed9553",
      "source": "jupyter-noop-example"
    },
    "id": "84465f35-67b2-4b48-8900-4366fd400854",
    "lastPaymentAt": null,
    "nextPaymentAt": null,
    "provider": "noop",
    "purchaseOrder": {
      "currency": "USD",
      "grossAmount": 3000,
      "items": [
        {
          "description": "Demo product paid through the noop provider",
          "extra": null,
          "image": null,
          "name": "Widget Pro",
      

'd86992bb-5349-4475-9282-7430ae369593'

## 6. Read Payment And Transactions

In [16]:
payment_after_pay = request("GET", f"/payments/{payment_id}")
assert payment_after_pay["status"] == "CLOSED"

transactions = request("GET", f"/payment-transactions?paymentId={payment_id}&page=0&size=10")
assert transactions["totalItems"] >= 1
assert any(item["id"] == payment_transaction_id for item in transactions["items"])

GET /payments/84465f35-67b2-4b48-8900-4366fd400854 -> 200
{
  "billingInfo": {
    "address1": "123 Market St",
    "address2": null,
    "city": "San Francisco",
    "country": "US",
    "email": "jane.doe@example.com",
    "firstName": "Jane",
    "lastName": "Doe",
    "phone": null,
    "postalCode": "94105",
    "state": null,
    "vatId": null
  },
  "createdAt": "2026-06-26T11:07:39.235963Z",
  "extra": {
    "externalOrderId": "demo-order-4a097865-5579-4278-adec-5d5e83ed9553",
    "source": "jupyter-noop-example"
  },
  "id": "84465f35-67b2-4b48-8900-4366fd400854",
  "lastPaymentAt": null,
  "nextPaymentAt": null,
  "provider": "noop",
  "purchaseOrder": {
    "currency": "USD",
    "grossAmount": 3000,
    "items": [
      {
        "description": "Demo product paid through the noop provider",
        "extra": null,
        "image": null,
        "name": "Widget Pro",
        "price": 3000,
        "quantity": 1,
        "sku": "WIDGET-PRO-DEMO",
        "tax": null,
        "

## 7. Idempotency Replay Check

Calling the same `/pay` operation with the same idempotency key returns the cached response. The payment itself is already closed, but the idempotency layer should replay the original response before the service attempts another payment.

In [17]:
replayed_response = request(
    "POST",
    f"/payments/{payment_id}/pay",
    headers={"Idempotency-Key": idempotency_key},
)

assert replayed_response["paymentTransaction"]["id"] == payment_transaction_id
assert replayed_response["paymentTransaction"]["status"] == "SUCCESS"

POST /payments/84465f35-67b2-4b48-8900-4366fd400854/pay -> 200
{
  "nextAction": null,
  "payment": {
    "billingInfo": {
      "address1": "123 Market St",
      "address2": null,
      "city": "San Francisco",
      "country": "US",
      "email": "jane.doe@example.com",
      "firstName": "Jane",
      "lastName": "Doe",
      "phone": null,
      "postalCode": "94105",
      "state": null,
      "vatId": null
    },
    "createdAt": 1782472059.235963,
    "extra": {
      "externalOrderId": "demo-order-4a097865-5579-4278-adec-5d5e83ed9553",
      "source": "jupyter-noop-example"
    },
    "id": "84465f35-67b2-4b48-8900-4366fd400854",
    "lastPaymentAt": null,
    "nextPaymentAt": null,
    "provider": "noop",
    "purchaseOrder": {
      "currency": "USD",
      "grossAmount": 3000,
      "items": [
        {
          "description": "Demo product paid through the noop provider",
          "extra": null,
          "image": null,
          "name": "Widget Pro",
          "price":

## Optional: Read RabbitMQ Debug Queue

RabbitMQ exchanges do not store messages by themselves. To inspect event payloads, create debug queues and bind them before running the payment flow.

Example bindings:

- queue `debug.payment.created` bound to exchange `payment.created` with routing key `payment.created`
- queue `debug.payment.finalized` bound to exchange `payment.finalized` with routing key `payment.finalized`
- queue `debug.payment.closed` bound to exchange `payment.closed` with routing key `payment.closed`

RabbitMQ Management UI: `http://localhost:15672`.

In [18]:
RABBITMQ_API = "http://localhost:15672/api"
RABBITMQ_AUTH = ("paymentgateway", "paymentgateway")
RABBITMQ_VHOST = "paymentgateway"

def rabbitmq_get_messages(queue_name, count=5):
    url = f"{RABBITMQ_API}/queues/{requests.utils.quote(RABBITMQ_VHOST, safe='')}/{queue_name}/get"
    response = requests.post(
        url,
        auth=RABBITMQ_AUTH,
        json={
            "count": count,
            "ackmode": "ack_requeue_true",
            "encoding": "auto",
            "truncate": 50000,
        },
        timeout=10,
    )
    print(f"RabbitMQ get {queue_name} -> {response.status_code}")
    if response.status_code == 404:
        print("Queue not found. Create and bind the debug queue before publishing events.")
        return []
    response.raise_for_status()
    messages = response.json()
    pretty(messages)
    return messages

rabbitmq_get_messages("debug.payment.finalized")

RabbitMQ get debug.payment.finalized -> 200
[]


[]